In [1]:
# CELL 1
!pip install -U transformers accelerate bitsandbytes
!pip install faster-whisper
!pip install python-docx
print("completed download")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 112.0 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
  

In [2]:
import base64
from IPython.display import Image, display

def render_mermaid(code):
    """Renders Mermaid.js code into an image within the notebook."""
    # Clean up the code in case Gemma added markdown backticks
    clean_code = code.replace('```mermaid', '').replace('```', '').strip()
    
    try:
        # Encode the code into a format the web service understands
        graphbytes = clean_code.encode("utf-8")
        base64_bytes = base64.b64encode(graphbytes)
        base64_string = base64_bytes.decode("ascii")
        
        # Display the resulting image
        display(Image(url="https://mermaid.ink/img/" + base64_string))
    except Exception as e:
        print(f"Could not render mindmap: {e}")
print("everything is fine")

everything is fine


In [3]:
# CELL 2
import os, gc, torch, pandas as pd
from faster_whisper import WhisperModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# Force PyTorch to manage memory better
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

clear_vram()
print("VRAM cleared. Ready for fresh start.")

VRAM cleared. Ready for fresh start.


In [4]:
# CELL 3
print("Loading Whisper...")
# Compute type float16 is the most efficient for T4 GPUs
whisper_model = WhisperModel("distil-large-v3", device="cuda", device_index=0, compute_type="float16")
print("Whisper loaded on GPU 0 ")

Loading Whisper...


Whisper loaded on GPU 0 


In [5]:
# CELL 4 
print("Loading Gemma 4...")
model_id = "google/gemma-4-E4B-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

gemma_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={"": 1}
)

tokenizer = AutoTokenizer.from_pretrained(model_id)


pipe = pipeline(
    "text-generation",
    model=gemma_model,
    tokenizer=tokenizer
)
print("Gemma 4 loaded is distributed on both gpu.")

Loading Gemma 4...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Gemma 4 loaded is distributed on both gpu.


In [6]:
# CELL 5
def transcribe_audio(audio_path):
  
    segments, info = whisper_model.transcribe(audio_path, beam_size=5, task="translate")
    return " ".join([segment.text for segment in segments])

def process_with_gemma(text, task_type):
    clear_vram()

    max_chunk_size = 6000 
    chunks = [text[i:i+max_chunk_size] for i in range(0, len(text), max_chunk_size)]
    
    all_results = []
    
   
    for chunk in chunks:
        if task_type == "flashcards":
            sys_msg = "Extract all the key definitions from the text. Output strictly as a JSON array: [{'q': 'question', 'a': 'answer'}]. No extra text and answer should be direct to point and bit detailed."
            task_temp = 0.1
        elif task_type == "notes":
            sys_msg = "Summarize the core concepts into concise bullet points of every important and small concepts too and just mention the diagram name if its mentioned in the lecture."
            task_temp = 0.4
        elif task_type == "mindmap":
            sys_msg = "Extract the core concepts and output strictly valid Mermaid.js mindmap syntax. You MUST start the response with the word 'mindmap', followed by the root node, and use spacing/indentation for child nodes. DO NOT output markdown formatting. DO NOT use classDiagram or flowchart syntax. Output ONLY the raw Mermaid code."
            task_temp = 0.1
            
        messages = [{"role": "user", "content": f"{sys_msg}\n\nText:\n{chunk}"}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        outputs = pipe(
            prompt,
            temperature=task_temp,
            do_sample=True,
            max_new_tokens=1024,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id 
        )
        
        result = outputs[0]['generated_text'][len(prompt):].strip()
        all_results.append(result)
        
    print(f"#{task_type.upper()} DONE")
    
  
    return "\n\n".join(all_results)#it will be combineing all the results
print("#DONE")

#DONE


In [7]:
# CELL 6
from docx import Document
import os

audio_files = ["/kaggle/input/datasets/harshitrajsrivastava/intotopsycology/1. Introduction.mp4"] 
results = []

for file in audio_files:
    print(f"--- Processing: {file} ---")
    try:
      
        transcript = transcribe_audio(file)
        notes = process_with_gemma(transcript, "notes")
        flashcards = process_with_gemma(transcript, "flashcards")
        mindmap_code = process_with_gemma(transcript, "mindmap") # NOW IT IS HERE
        
     
        results.append({
            "file": file,
            "transcript": transcript,
            "notes": notes,
            "flashcards": flashcards,
            "mindmap": mindmap_code
        })
        
    
        print("\n...sab changa si...")
        print("\n[Generated Mindmap]:")
        
        for part in mindmap_code.split("mindmap"):
            if part.strip():
                render_mermaid("mindmap " + part)
    except Exception as e:
        print(f"Error processing {file}: {e}")

print("Generating Word Document...")
doc = Document()
doc.add_heading('Video Processing Results', 0)

for res in results:
    
    doc.add_heading(f"Source: {os.path.basename(res['file'])}", level=1)
    
    
    doc.add_heading('Notes', level=2)
    doc.add_paragraph(res['notes'])
    
   
    doc.add_heading('Flashcards', level=2)
    doc.add_paragraph(res['flashcards'])
    
    doc.add_heading('Mindmap (Raw Code)', level=2)
    doc.add_paragraph(res['mindmap'])
    
    doc.add_page_break()


doc.save("Submission.docx")
print("\nSuccess! Open your Kaggle output folder to download Submission.docx")

--- Processing: /kaggle/input/datasets/harshitrajsrivastava/intotopsycology/1. Introduction.mp4 ---


[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'pad_token_id', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force clean

#NOTES DONE


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

#FLASHCARDS DONE


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

#MINDMAP DONE

...sab changa si...

[Generated Mindmap]:


Generating Word Document...

Success! Open your Kaggle output folder to download Submission.docx
